In [2]:
import cv2

In [18]:
!wget http://download.tensorflow.org/models/object_detection/mask_rcnn_inception_v2_coco_2018_01_28.tar.gz
!tar zxvf mask_rcnn_inception_v2_coco_2018_01_28.tar.gz

--2022-04-12 12:23:17--  http://download.tensorflow.org/models/object_detection/mask_rcnn_inception_v2_coco_2018_01_28.tar.gz
Resolving download.tensorflow.org (download.tensorflow.org)... 2a00:1450:400e:80e::2010, 142.251.36.16
Connecting to download.tensorflow.org (download.tensorflow.org)|2a00:1450:400e:80e::2010|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 177817887 (170M) [application/x-tar]
Saving to: ‘mask_rcnn_inception_v2_coco_2018_01_28.tar.gz’

mask_rcnn_inception 100%[===================>] 169,58M  8,83MB/s    in 25s     

2022-04-12 12:23:42 (6,91 MB/s) - ‘mask_rcnn_inception_v2_coco_2018_01_28.tar.gz’ saved [177817887/177817887]

x mask_rcnn_inception_v2_coco_2018_01_28/
x mask_rcnn_inception_v2_coco_2018_01_28/model.ckpt.index
x mask_rcnn_inception_v2_coco_2018_01_28/checkpoint
x mask_rcnn_inception_v2_coco_2018_01_28/pipeline.config
x mask_rcnn_inception_v2_coco_2018_01_28/model.ckpt.data-00000-of-00001
x mask_rcnn_inception_v2_coco_2018_01_2

In [1]:
# Initialize the parameters
confThreshold = 0.5  #Confidence threshold
maskThreshold = 0.3  # Mask threshold

In [20]:
# Load names of classes
classesFile = "mscoco_labels.names";
classes = None
with open(classesFile, 'rt') as f:
   classes = f.read().rstrip('\n').split('\n')

# Load the colors
colorsFile = "colors.txt";
with open(colorsFile, 'rt') as f:
    colorsStr = f.read().rstrip('\n').split('\n')
colors = []
for i in range(len(colorsStr)):
    rgb = colorsStr[i].split(' ')
    color = np.array([float(rgb[0]), float(rgb[1]), float(rgb[2])])
    colors.append(color)

# Give the textGraph and weight files for the model
textGraph = "./mask_rcnn_inception_v2_coco_2018_01_28.pbtxt";
modelWeights = "./mask_rcnn_inception_v2_coco_2018_01_28/frozen_inference_graph.pb";

# Load the network
net = cv.dnn.readNetFromTensorflow(modelWeights, textGraph);
net.setPreferableBackend(cv.dnn.DNN_BACKEND_OPENCV)
net.setPreferableTarget(cv.dnn.DNN_TARGET_CPU)

FileNotFoundError: [Errno 2] No such file or directory: 'mscoco_labels.names'

In [15]:
net = cv2.dnn.readNetFromTensorflow("assets/models/mask_rcnn_inception_v2_coco_2018_01_28/frozen_inference_graph.pb",
)

In [16]:
img = cv2.imread("assets/images/pixel_transfers/db/Magritte_TheSonOfMan.jpg")
# Detect objects
blob = cv2.dnn.blobFromImage(img, swapRB=True, crop=False)
net.setInput(blob)

In [17]:
boxes, masks = net.forward(["detection_out_final", "detection_masks"])
detection_count = boxes.shape[2]

error: OpenCV(4.5.1) ../modules/dnn/src/dnn.cpp:794: error: (-215:Assertion failed) inputs.size() == requiredOutputs in function 'getMemoryShapes'


In [ ]:
    # Create a 4D blob from a frame.

    blob = cv.dnn.blobFromImage(frame, swapRB=True, crop=False)

    # Set the input to the network
    net.setInput(blob)


In [3]:
import cv2
import numpy as np
path_to_frozen_inference_graph = 'assets/models/mask_rcnn_inception_v2_coco_2018_01_28/frozen_inference_graph.pb'
path_coco_model= 'assets/models/mask_rcnn_inception_v2_coco_2018_01_28/mask_rcnn_inception_v2_coco_2018_01_28.pbtxt'
VIDEO = '/Users/derrickvanfrausum/BeCode_AI/git-repos/learn-ai/opencv/assets/videos/P1322115.MP4'
net = cv2.dnn.readNetFromTensorflow(path_to_frozen_inference_graph,path_coco_model)
colors = np.random.randint(125, 255, (80, 3))
video = cv2.VideoCapture(VIDEO)
while True:
    grabbed,frame=video.read()
    if not grabbed:
        break
    # img=cv2.resize(frame,(650,550))
    img = frame.copy()
    height, width, _ = img.shape
    black_image = np.zeros((height, width, 3), np.uint8)
    black_image[:] = (0, 0, 0)
    blob = cv2.dnn.blobFromImage(img, swapRB=True)
    net.setInput(blob)
    boxes, masks = net.forward(["detection_out_final", "detection_masks"])
    detection_count = boxes.shape[2]
    for i in range(detection_count):
        box = boxes[0, 0, i]
        class_id = box[1]
        score = box[2]
        if score < 0.5:
            continue
        x = int(box[3] * width)
        y = int(box[4] * height)
        x2 = int(box[5] * width)
        y2 = int(box[6] * height)
        roi = black_image[y: y2, x: x2]
        roi_height, roi_width, _ = roi.shape
        mask = masks[i, int(class_id)]
        mask = cv2.resize(mask, (roi_width, roi_height))
        _, mask = cv2.threshold(mask, 0.5, 255, cv2.THRESH_BINARY)
        cv2.rectangle(img, (x, y), (x2, y2), (255, 0, 0), 3)
        contours, _ = cv2.findContours(np.array(mask, np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        color = colors[int(class_id)]
        for cnt in contours:
            cv2.fillPoly(roi, [cnt], (int(color[0]), int(color[1]), int(color[2])))
    cv2.imshow("Black image", black_image)
    final_frame = ((0.6*black_image)+(0.4*frame)).astype("uint8")
    cv2.imshow("Overlay Frames",final_frame)
    key = cv2.waitKey(1) & 0xFF
    if key == ord("q"):
             break
video.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

KeyboardInterrupt: 

In [4]:
cv2.destroyAllWindows()
cv2.waitKey(1)

-1